# 🔴 Interview: Mamba SSM Step

Selective state-space model with input-dependent B, C, and delta

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def mamba_ssm_step(x, h, A, B, C, delta):
    """
    手写 Mamba SSM 单步递推 —— 面试版。

    面试考点:
    1. Mamba vs 经典 SSM: B, C, delta 是输入相关的 (选择性)
    2. 零阶保持离散化: dA = exp(δ*A), dB = δ*B
    3. 递推公式: h_new = dA*h + dB*x, y = (h_new * C).sum(-1)
    4. A 通常为负值 (稳定), delta 经过 softplus (正值)

    参数:
        x: (B, D), h: (B, D, N), A: (D, N), B: (B, D, N), C: (B, D, N), delta: (B, D)
    返回: (y, h_new)
    """
    # ---- Step 1: 离散化 ----
    # dA = exp(δ * A), dB = δ * B
    # δ 从 (B,D) 扩展到 (B,D,1) 与 A(D,N) 和 B(B,D,N) 广播
    dA = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0))  # (B, D, N)
    dB = delta.unsqueeze(-1) * B                           # (B, D, N)

    # ---- Step 2: 状态更新 ----
    # h_new = dA * h + dB * x
    # dA 作为衰减因子 (<1 因为 A<0), dB 控制新信息写入量
    h_new = dA * h + dB * x.unsqueeze(-1)  # (B, D, N)

    # ---- Step 3: 输出 ----
    # y = Σ_n h_new * C  (C 选择性地读取状态)
    y = (h_new * C).sum(dim=-1)  # (B, D)

    return y, h_new

In [ ]:
torch.manual_seed(0)
B_size, D, N = 2, 8, 4  # batch, channels, state_dim

x     = torch.randn(B_size, D)
h     = torch.zeros(B_size, D, N)
A     = -torch.rand(D, N)          # negative for stability
B_mat = torch.randn(D, N)
C     = torch.randn(B_size, D, N)
delta = torch.rand(B_size, D).add(0.1)  # positive step sizes

y, h_new = mamba_ssm_step(x, h, A, B_mat, C, delta)

print("y shape:    ", y.shape)      # (2, 8)
print("h_new shape:", h_new.shape)  # (2, 8, 4)

# Verify dA = exp(delta * A) numerically for first element
dA_manual = torch.exp(delta[0, 0] * A[0])
dA_check  = torch.exp(delta[0:1, 0:1] * A[0:1])[0, 0]
print("dA formula check (should be ~0):", (dA_manual - dA_check).abs().max().item())

In [ ]:
from torch_judge import check
check("mamba_ssm")

## 📝 核心思路总结

1. **选择性是 Mamba 的灵魂**：B、C、delta 都是输入相关的，模型可以根据当前输入决定「写入什么、读取什么、遗忘多快」。
2. **零阶保持离散化**：`dA = exp(δ*A)`，`dB = δ*B`，将连续 SSM 离散化；A<0 保证 dA<1（衰减），δ>0 保证离散化有效。
3. **递推公式**：`h_new = dA * h + dB * x`，dA 是衰减因子（遗忘），dB*x 是新信息（记忆），y = (h_new * C).sum(-1) 是选择性读取。
4. **并行化**：虽然形式上是递推，但 dA 的结构允许用并行扫描 (parallel scan) 高效计算，这是 Mamba 比 Transformer 快的关键。